# 09 | LangGraph 状态更新与 Reducer

> 本文示例基于 LangGraph 1.2.4。

LangGraph 的节点返回的是对 State 的**更新值**。Reducer 决定如何把更新值写回状态：

```text
旧状态 + 节点更新值 → Reducer → 新状态
```

> Reducer 由 LangGraph 在执行图时调用。直接调用普通节点函数只会得到它返回的字典，不会触发状态合并。

Reducer 很重要，因为它明确了每个字段的更新规则：是覆盖旧值、累加数据、去重合并，还是追加消息。多个节点同时更新同一字段时，LangGraph 也需要 reducer 来决定如何合并这些更新，否则可能出现并发更新冲突。

常见作用包括：

- 保存字段的最新值；
- 累积多个节点产生的结果；
- 合并并去除重复数据；
- 保存完整的对话消息历史；
- 为并行节点提供明确的状态合并规则。

## 一、默认覆盖（无显式 reducer）

- **类别**：默认覆盖更新，字段没有通过 `Annotated` 声明 reducer。
- **场景**：只需要保存最新值，例如当前步骤、运行状态、最终答案。
- **效果**：节点返回的新值直接替换旧值。节点应返回最终值，而不是增量。


In [ ]:
from typing import TypedDict

from langgraph.graph import START, END, StateGraph


class State(TypedDict):
    count: int  # 未声明 reducer，默认用新值覆盖旧值


def increment(state: State):
    # 没有 reducer，因此节点返回计算后的最终值
    return {"count": state["count"] + 1}


builder = StateGraph(State)
builder.add_node("increment", increment)
builder.add_edge(START, "increment")
builder.add_edge("increment", END)
graph = builder.compile()

print(graph.invoke({"count": 1}))  # {'count': 2}

## 二、累加型 reducer：`operator.add`

- **类别**：累加型 reducer，使用 Python 标准库的 `operator.add`。
- **场景**：累积列表、数字或字符串，例如收集多个节点的结果。
- **效果**：LangGraph 执行 `旧值 + 节点更新值`。列表会拼接，数字会相加。

> 节点只需返回本次新增的数据，不要再次手动合并旧值。


In [ ]:
import operator
from typing import Annotated, TypedDict

from langgraph.graph import START, END, StateGraph


class State(TypedDict):
    # operator.add 将节点返回的新列表拼接到旧列表后面
    items: Annotated[list[str], operator.add]


def add_first_item(state: State):
    return {"items": ["Apple"]}  # 只返回本节点新增的元素


def add_second_item(state: State):
    return {"items": ["Banana"]}

builder = StateGraph(State)
builder.add_node("first", add_first_item)
builder.add_node("second", add_second_item)

builder.add_edge(START, "first")
builder.add_edge("first", "second")
builder.add_edge("second", END)

graph = builder.compile()
print(graph.invoke({"items": []}))  # {'items': ['Apple', 'Banana']}


## 三、字典合并型 reducer：`operator.or_`

- **类别**：合并型 reducer，使用 Python 字典的 `|` 运算。
- **场景**：多个节点逐步补充元数据、配置或结构化结果。
- **效果**：合并新旧字典；key 相同时，节点返回的新值覆盖旧值。


In [ ]:
import operator
from typing import Annotated, TypedDict

from langgraph.graph import START, END, StateGraph


class State(TypedDict):
    # operator.or_ 等价于：旧字典 | 更新字典
    data: Annotated[dict[str, str], operator.or_]


def add_result(state: State):
    return {"data": {"status": "done", "owner": "Bob"}}


builder = StateGraph(State)
builder.add_node("add_result", add_result)
builder.add_edge(START, "add_result")
builder.add_edge("add_result", END)
graph = builder.compile()

initial_state: State = {"data": {"task": "report", "status": "pending"}}
print(graph.invoke(initial_state))


## 四、自定义 reducer：合并并去重

- **类别**：自定义合并型 reducer。
- **场景**：内置规则无法满足需求，例如标签、搜索结果需要累积但不能重复。
- **效果**：合并旧列表和节点更新列表，去除重复项，并保留第一次出现的顺序。


In [ ]:
from typing import Annotated, TypedDict

from langgraph.graph import START, END, StateGraph


def merge_unique(
    current: list[str],
    update: list[str],
) -> list[str]:
    # 先拼接新旧列表，再利用字典 key 去重并保留原顺序
    return list(dict.fromkeys(current + update))


class State(TypedDict):
    # 更新 tags 时，LangGraph 调用 merge_unique(旧值, 更新值)
    tags: Annotated[list[str], merge_unique]


def add_tags(state: State):
    # Banana 已存在，经过 reducer 后不会重复
    return {"tags": ["Banana", "Orange"]}


builder = StateGraph(State)
builder.add_node("add_tags", add_tags)

builder.add_edge(START, "add_tags")
builder.add_edge("add_tags", END)

graph = builder.compile()
result = graph.invoke({
    "tags": ["Apple", "Banana"]
})

print(result)  # {'tags': ['Apple', 'Banana', 'Orange']}

## 五、MessagesState 中的 reducer

- **类别**：LangGraph 内置的消息专用 reducer：`add_messages`。
- **场景**：聊天机器人、工具调用 Agent 等需要持续保存消息历史的工作流。
- **效果**：把节点返回的新消息加入已有历史；如果消息 `id` 相同，则用新消息替换旧消息。

`MessagesState` 已经内置这个 reducer。下面将它显式声明出来，便于观察 `HumanMessage` 和 `AIMessage` 的合并过程。


In [ ]:
from typing import Annotated, TypedDict

from langchain_core.messages import AIMessage, AnyMessage, HumanMessage
from langgraph.graph import START, END, StateGraph
from langgraph.graph.message import add_messages


class State(TypedDict):
    # add_messages 追加新消息；消息 id 相同时则替换旧消息
    messages: Annotated[list[AnyMessage], add_messages]


def reply(state: State):
    # 只返回本次新增的 AI 消息，不需要手动拼接历史
    return {"messages": [AIMessage(content="有什么可以帮助你的？")]}


builder = StateGraph(State)
builder.add_node("reply", reply)
builder.add_edge(START, "reply")
builder.add_edge("reply", END)

graph = builder.compile()

# 图执行前已经有一条用户消息
initial_state: State = {
    "messages": [HumanMessage(content="你好")]
}

result = graph.invoke(initial_state)  # reducer 合并旧消息和新消息

print("节点执行前：", [m.content for m in initial_state["messages"]])
print("节点只新增：", ["有什么可以帮助你的？"])
print("Reducer 合并后：", [m.content for m in result["messages"]])

## 六、小结

Reducer 是针对 **State 中的字段**声明的，不是为整个 State 统一设置。选择规则时，先判断该字段需要保存最终值，还是累积每次更新。

| 更新需求 | Reducer | 效果 |
| --- | --- | --- |
| 保存最新值 | 不声明 reducer | 新值覆盖旧值 |
| 累加数字或列表 | `operator.add` | 执行 `旧值 + 更新值` |
| 合并字典 | `operator.or_` | 合并 key，重复 key 使用新值 |
| 实现业务合并规则 | 自定义函数 | 例如合并并去重 |
| 保存消息历史 | `add_messages` | 追加消息，相同 ID 时替换 |


## 七、延伸补充

本节补充示例代码中涉及的 Python 概念。这些内容并非 LangGraph 特有，但理解它们有助于读懂 reducer 的声明和工作方式，也可以作为延伸学习。

### 1. `operator` 标准库

`operator` 把运算符包装成函数。LangGraph 可以把这些函数作为 reducer：

In [ ]:
import operator

# operator.add(a, b) 等价于 a + b，行为由数据类型决定
print(operator.add(1, 2))          # 数字相加：3
print(operator.add(["A"], ["B"]))  # 列表拼接：['A', 'B']

### 2. `Annotated` 附加元数据

`Annotated` 本身不执行规则，只负责给类型附加元数据。LangGraph 会读取其中的 reducer。

In [ ]:
from typing import Annotated, get_args

# int 是真实类型，字符串是附加元数据
Age = Annotated[int, "必须大于 10"]

# Python 不会自动执行附加规则，所以这里仍然可以赋值为 0
age: Age = 0
print("age:", age)

# 手动读取 Annotated 信息
real_type, rule = get_args(Age)

print("真实类型:", real_type)
print("附加信息:", rule)
